# Finetuning実験 Part 3: 評価・比較分析

**目的**: RAGとファインチューニングの性能を比較評価

**比較対象**:
1. **Baseline**: Qwen2.5-7B-Instruct（RAGなし）
2. **RAG**: Qwen2.5-7B-Instruct + 構造化RAG（Phase 6）
3. **FT-Base**: ファインチューニングモデル（RAGなし）
4. **FT+RAG**: ファインチューニングモデル + 構造化RAG

**評価**:
- 55テストケース（L1-L5）
- evaluators_v2.pyの評価関数

**作成日**: 2026-01-28

## Section 1: 環境セットアップ

In [ ]:
# 1.1 パッケージインストール
%%capture
!pip install -q torch transformers accelerate bitsandbytes
!pip install -q peft
!pip install -q langchain langchain-community langchain-huggingface langchain-chroma
!pip install -q chromadb sentence-transformers
!pip install -q pandas numpy tqdm matplotlib seaborn scipy
print("パッケージインストール完了")

In [ ]:
# 1.2 GPU確認・メモリ管理
import torch
import gc

def print_memory():
    if torch.cuda.is_available():
        allocated = torch.cuda.memory_allocated() / 1e9
        reserved = torch.cuda.memory_reserved() / 1e9
        print(f"GPU VRAM: {allocated:.2f}GB allocated, {reserved:.2f}GB reserved")
    import psutil
    print(f"RAM: {psutil.virtual_memory().percent}%")

def clear_memory():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print_memory()

In [ ]:
# 1.3 Google Driveマウント
from google.colab import drive
drive.mount('/content/drive')

import os, sys
BASE_DIR = "/content/drive/MyDrive/experiments-local-llm"
DATA_DIR = f"{BASE_DIR}/data"
MODEL_DIR = f"{BASE_DIR}/models"
RESULTS_DIR = f"{BASE_DIR}/results"
os.makedirs(RESULTS_DIR, exist_ok=True)
sys.path.insert(0, BASE_DIR)
print(f"BASE_DIR: {BASE_DIR}")

In [ ]:
# 1.4 設定
BASE_MODEL = "Qwen/Qwen2.5-7B-Instruct"
FT_MODEL_DIR = f"{MODEL_DIR}/qwen2.5-7b-shibuya-poi"
EMBEDDING_MODEL = "intfloat/multilingual-e5-base"

print(f"Base Model: {BASE_MODEL}")
print(f"FT Model: {FT_MODEL_DIR}")
print(f"Embedding: {EMBEDDING_MODEL}")

## Section 2: データ・モジュール読み込み

In [ ]:
# 2.1 POIデータ読み込み
import json
from collections import Counter

with open(f"{DATA_DIR}/poi_documents.json", "r", encoding="utf-8") as f:
    poi_documents = json.load(f)

all_pois = []
for doc in poi_documents:
    poi = doc["metadata"].copy() if "metadata" in doc else doc.copy()
    poi["content"] = doc.get("content", "")
    all_pois.append(poi)
    
print(f"POIデータ: {len(all_pois)}件")

In [ ]:
# 2.2 空間情報追加（geo_utilsから）
from src.geo_utils import enrich_all_pois, get_nearest_pois, filter_by_radius, compare_by_radius
from src.geo_utils import generate_proximity_context, generate_sensitivity_context
from src.aggregator import compare_east_west, get_top_categories, analyze_category_by_direction, filter_by_category
from src.structured_rag_system import analyze_question

enriched_pois = enrich_all_pois(all_pois)
print(f"空間情報追加完了: {len(enriched_pois)}件")

In [ ]:
# 2.3 テストケース読み込み
from src.test_cases_v2 import TEST_CASES_V2, get_test_cases_by_level, get_test_cases_by_subcategory
print(f"テストケース: {len(TEST_CASES_V2)}件")

# レベル別件数
level_counts = Counter(tc.level for tc in TEST_CASES_V2)
print("\nレベル別:")
for level in sorted(level_counts.keys()):
    print(f"  L{level}: {level_counts[level]}件")

## Section 3: モデルセットアップ

In [ ]:
# 3.1 Embeddingモデル（RAG用）
from langchain_huggingface import HuggingFaceEmbeddings

print("Embeddingモデルロード中...")
embeddings = HuggingFaceEmbeddings(
    model_name=EMBEDDING_MODEL,
    model_kwargs={'device': DEVICE},
    encode_kwargs={'normalize_embeddings': True}
)
print("完了")
print_memory()

In [ ]:
# 3.2 ベクトルストア構築
from langchain_chroma import Chroma
from langchain_core.documents import Document

def flatten_metadata(metadata):
    flat = {}
    for key, value in metadata.items():
        if isinstance(value, dict):
            flat[key] = json.dumps(value, ensure_ascii=False)
        elif isinstance(value, list):
            flat[key] = json.dumps(value, ensure_ascii=False)
        elif value is None:
            flat[key] = ""
        elif isinstance(value, (str, int, float, bool)):
            flat[key] = value
        else:
            flat[key] = str(value)
    return flat

print("ベクトルストア構築中...")
documents = []
for poi in poi_documents:
    if "metadata" in poi:
        content = poi.get("content", f"{poi['metadata'].get('name', '')}")
        metadata = flatten_metadata(poi["metadata"])
    else:
        content = poi.get("content", f"{poi.get('name', '')}")
        metadata = flatten_metadata(poi)
    if "content" in metadata:
        del metadata["content"]
    documents.append(Document(page_content=content, metadata=metadata))

vectorstore = Chroma.from_documents(
    documents=documents,
    embedding=embeddings,
    collection_name="poi_eval"
)
print(f"完了: {len(documents)}件")
print_memory()

In [ ]:
# 3.3 Embeddingモデル解放
del embeddings
clear_memory()
print_memory()

In [ ]:
# 3.4 ベースモデルロード（量子化）
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

print(f"ベースモデルロード中: {BASE_MODEL}")
base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
    low_cpu_mem_usage=True
)
print("完了")
print_memory()

In [ ]:
# 3.5 ファインチューニングモデルロード
from peft import PeftModel

print(f"ファインチューニングモデルロード中: {FT_MODEL_DIR}")

# ファインチューニングモデルが存在するか確認
if os.path.exists(FT_MODEL_DIR):
    # LoRAアダプタを適用
    ft_model = PeftModel.from_pretrained(
        base_model,
        FT_MODEL_DIR
    )
    ft_tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_DIR)
    FT_AVAILABLE = True
    print("ファインチューニングモデルロード完了")
else:
    print(f"警告: {FT_MODEL_DIR} が見つかりません")
    print("ファインチューニングなしで評価を実行します")
    ft_model = None
    ft_tokenizer = None
    FT_AVAILABLE = False

print_memory()

## Section 4: 評価システム定義

In [ ]:
# 4.1 システムプロンプト
SYSTEM_PROMPT = """あなたは渋谷エリアの地理情報に詳しいアシスタントです。
提供された情報に基づいて、正確かつ簡潔に回答してください。
座標情報がある場合は必ず含めてください。
数値データがある場合は具体的な数字を使って回答してください。"""

In [ ]:
# 4.2 推論関数
import time

def generate_response(model, tokenizer, question, context=None, max_tokens=512):
    """モデルで回答を生成"""
    start_time = time.time()
    
    if context:
        user_content = f"""以下の情報を参考にして質問に回答してください。

{context}

【質問】
{question}

【回答】
上記の情報を基に、具体的な数値や座標を含めて回答します。"""
    else:
        user_content = question
    
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_content}
    ]
    
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )
    
    inputs = tokenizer(text, return_tensors="pt").to(DEVICE)
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_tokens,
            temperature=0.1,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )
    
    response = tokenizer.decode(outputs[0], skip_special_tokens=True)
    if "assistant" in response.lower():
        response = response.split("assistant")[-1].strip()
    
    elapsed = time.time() - start_time
    
    return response, elapsed

In [ ]:
# 4.3 RAGコンテキスト構築
def build_rag_context(question, vectorstore, all_pois):
    """構造化RAGコンテキストを構築（Phase 6.2.1）"""
    analysis = analyze_question(question)
    structured_parts = []
    
    cat = analysis.subcategories[0] if analysis.subcategories else None
    
    # 近接性検索
    if analysis.requires_proximity and cat:
        structured_parts.append(generate_proximity_context(all_pois, cat, top_n=5))
    
    # 感度分析
    if analysis.requires_sensitivity and cat:
        r1, r2 = analysis.sensitivity_radii if analysis.sensitivity_radii else (300, 500)
        structured_parts.append(generate_sensitivity_context(all_pois, cat, r1, r2))
    
    # 東西比較
    if analysis.requires_comparison and "東" in question and "西" in question:
        result = compare_east_west(all_pois, cat)
        structured_parts.append(f"【東西比較】\n{result.to_japanese()}")
    
    # 集計
    if analysis.requires_aggregation:
        if cat:
            filtered = filter_by_category(all_pois, cat)
            structured_parts.append(f"【{cat}の集計】\n総数: {len(filtered)}件")
        else:
            top = get_top_categories(all_pois, 5)
            structured_parts.append("【カテゴリランキング】")
            for i, c in enumerate(top, 1):
                structured_parts.append(f"  {i}. {c.category}: {c.count}件")
    
    # ベクトル検索（常に追加）
    try:
        results = vectorstore.similarity_search(question, k=5)
        if results:
            lines = ["【関連POI情報】"]
            for r in results:
                name = r.metadata.get('name', '不明')
                category = r.metadata.get('category', '')
                lat = r.metadata.get('lat', '')
                lon = r.metadata.get('lon', '')
                line = f"- {name}"
                if category:
                    line += f" ({category})"
                if lat and lon:
                    line += f" 座標: ({lat}, {lon})"
                lines.append(line)
            
            if structured_parts:
                structured_parts.append("")
            structured_parts.append("\n".join(lines))
    except Exception as e:
        print(f"ベクトル検索エラー: {e}")
    
    return "\n".join(structured_parts)

In [ ]:
# 4.4 評価関数
import re

def evaluate_response(answer, tc):
    """回答を評価してスコアを計算"""
    # キーワードヒット率
    keywords = tc.expected_keywords or []
    kw_hits = sum(1 for kw in keywords if kw.lower() in answer.lower())
    kw_score = (kw_hits / len(keywords) * 100) if keywords else 50
    
    # 座標情報の有無
    coord_score = 100 if ("35." in answer and "139." in answer) else 0
    
    # 数値の有無
    num_score = 100 if re.findall(r'\d+', answer) else 0
    
    # POI名の有無
    poi_score = 100 if any(kw in answer for kw in keywords if len(kw) > 2) else 0
    
    # レベル別の重み付け
    level = tc.level
    if level == 1:
        total = kw_score * 0.4 + coord_score * 0.3 + poi_score * 0.3
    elif level == 2:
        total = kw_score * 0.3 + coord_score * 0.2 + num_score * 0.5
    elif level == 3:
        total = kw_score * 0.2 + poi_score * 0.2 + num_score * 0.4 + coord_score * 0.2
    elif level == 4:
        total = kw_score * 0.3 + num_score * 0.4 + poi_score * 0.3
    else:  # level 5
        total = kw_score * 0.4 + num_score * 0.3 + poi_score * 0.3
    
    return {
        "keyword_score": round(kw_score, 1),
        "coord_score": coord_score,
        "number_score": num_score,
        "poi_score": poi_score,
        "total_score": round(total, 1)
    }

## Section 5: 評価実行

In [ ]:
# 5.1 評価ループ
from tqdm import tqdm

def run_evaluation(model, tokenizer, test_cases, use_rag=False, model_name="model"):
    """指定モデルで全テストケースを評価"""
    results = []
    
    for tc in tqdm(test_cases, desc=f"{model_name}"):
        try:
            # コンテキスト構築
            context = build_rag_context(tc.prompt, vectorstore, enriched_pois) if use_rag else None
            
            # 回答生成
            answer, elapsed = generate_response(model, tokenizer, tc.prompt, context)
            
            # 評価
            scores = evaluate_response(answer, tc)
            
            results.append({
                "id": tc.id,
                "level": tc.level,
                "subcategory": tc.subcategory,
                "prompt": tc.prompt,
                "answer": answer[:500],
                "time_sec": round(elapsed, 2),
                "scores": scores
            })
            
            clear_memory()
            
        except Exception as e:
            print(f"Error ({tc.id}): {e}")
            results.append({
                "id": tc.id,
                "level": tc.level,
                "error": str(e)
            })
    
    return results

print("評価関数定義完了")

In [ ]:
# 5.2 ベースライン評価（RAGなし）
print("=== ベースライン評価（RAGなし） ===")
baseline_results = run_evaluation(
    base_model, base_tokenizer, TEST_CASES_V2,
    use_rag=False, model_name="Baseline"
)
print(f"完了: {len(baseline_results)}件")

In [ ]:
# 5.3 RAG評価
print("=== RAG評価 ===")
rag_results = run_evaluation(
    base_model, base_tokenizer, TEST_CASES_V2,
    use_rag=True, model_name="RAG"
)
print(f"完了: {len(rag_results)}件")

In [ ]:
# 5.4 ファインチューニングモデル評価（RAGなし）
if FT_AVAILABLE:
    print("=== FT-Base評価（RAGなし） ===")
    ft_base_results = run_evaluation(
        ft_model, ft_tokenizer, TEST_CASES_V2,
        use_rag=False, model_name="FT-Base"
    )
    print(f"完了: {len(ft_base_results)}件")
else:
    print("FTモデルなし - スキップ")
    ft_base_results = None

In [ ]:
# 5.5 ファインチューニングモデル + RAG評価
if FT_AVAILABLE:
    print("=== FT+RAG評価 ===")
    ft_rag_results = run_evaluation(
        ft_model, ft_tokenizer, TEST_CASES_V2,
        use_rag=True, model_name="FT+RAG"
    )
    print(f"完了: {len(ft_rag_results)}件")
else:
    print("FTモデルなし - スキップ")
    ft_rag_results = None

## Section 6: 結果分析

In [ ]:
# 6.1 分析関数
import pandas as pd

def analyze_results(results, name):
    """結果を分析"""
    if results is None:
        return None
    
    valid = [r for r in results if "error" not in r]
    
    # 全体スコア
    all_scores = [r["scores"]["total_score"] for r in valid]
    overall_avg = sum(all_scores) / len(all_scores) if all_scores else 0
    
    # レベル別
    by_level = {}
    for level in [1, 2, 3, 4, 5]:
        level_results = [r for r in valid if r["level"] == level]
        if level_results:
            scores = [r["scores"]["total_score"] for r in level_results]
            by_level[level] = round(sum(scores) / len(scores), 1)
    
    # サブカテゴリ別
    by_subcategory = {}
    for r in valid:
        sc = r.get("subcategory", "unknown")
        if sc not in by_subcategory:
            by_subcategory[sc] = []
        by_subcategory[sc].append(r["scores"]["total_score"])
    by_subcategory = {k: round(sum(v)/len(v), 1) for k, v in by_subcategory.items()}
    
    # 平均時間
    times = [r["time_sec"] for r in valid if "time_sec" in r]
    avg_time = round(sum(times) / len(times), 2) if times else 0
    
    return {
        "name": name,
        "overall": round(overall_avg, 1),
        "by_level": by_level,
        "by_subcategory": by_subcategory,
        "avg_time_sec": avg_time,
        "valid_count": len(valid),
        "error_count": len(results) - len(valid)
    }

# 各モデルの分析
analysis_baseline = analyze_results(baseline_results, "Baseline")
analysis_rag = analyze_results(rag_results, "RAG")
analysis_ft_base = analyze_results(ft_base_results, "FT-Base") if ft_base_results else None
analysis_ft_rag = analyze_results(ft_rag_results, "FT+RAG") if ft_rag_results else None

print("分析完了")

In [ ]:
# 6.2 全体比較表
print("=" * 60)
print("全体スコア比較")
print("=" * 60)

models = [("Baseline", analysis_baseline), ("RAG", analysis_rag)]
if analysis_ft_base:
    models.append(("FT-Base", analysis_ft_base))
if analysis_ft_rag:
    models.append(("FT+RAG", analysis_ft_rag))

print(f"{'Model':<15} {'Score':>10} {'Time(s)':>10} {'Valid':>8}")
print("-" * 50)
for name, analysis in models:
    if analysis:
        print(f"{name:<15} {analysis['overall']:>10.1f} {analysis['avg_time_sec']:>10.2f} {analysis['valid_count']:>8}")

In [ ]:
# 6.3 レベル別比較
print("\n" + "=" * 60)
print("レベル別スコア比較")
print("=" * 60)

header = f"{'Level':<8}"
for name, _ in models:
    header += f"{name:>12}"
print(header)
print("-" * (8 + 12 * len(models)))

for level in [1, 2, 3, 4, 5]:
    row = f"L{level:<7}"
    for _, analysis in models:
        if analysis and level in analysis['by_level']:
            row += f"{analysis['by_level'][level]:>12.1f}"
        else:
            row += f"{'N/A':>12}"
    print(row)

In [ ]:
# 6.4 可視化
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 全体スコア棒グラフ
ax1 = axes[0]
model_names = [name for name, a in models if a]
scores = [a['overall'] for _, a in models if a]
bars = ax1.bar(model_names, scores, color=['#3498db', '#2ecc71', '#e74c3c', '#9b59b6'][:len(scores)])
ax1.set_ylabel('Score')
ax1.set_title('Overall Score Comparison')
ax1.set_ylim(0, 100)
for bar, score in zip(bars, scores):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, f'{score:.1f}', 
             ha='center', va='bottom', fontsize=10)

# レベル別レーダーチャート風の棒グラフ
ax2 = axes[1]
levels = [f'L{i}' for i in [1, 2, 3, 4, 5]]
x = range(len(levels))
width = 0.2

for i, (name, analysis) in enumerate(models):
    if analysis:
        level_scores = [analysis['by_level'].get(j, 0) for j in [1, 2, 3, 4, 5]]
        ax2.bar([xi + i*width for xi in x], level_scores, width, label=name, alpha=0.8)

ax2.set_xticks([xi + width*(len(models)-1)/2 for xi in x])
ax2.set_xticklabels(levels)
ax2.set_ylabel('Score')
ax2.set_title('Score by Level')
ax2.set_ylim(0, 100)
ax2.legend()

plt.tight_layout()
plt.savefig(f"{RESULTS_DIR}/finetuning_comparison.png", dpi=150)
plt.show()

print(f"図保存: {RESULTS_DIR}/finetuning_comparison.png")

In [ ]:
# 6.5 仮説検証
print("\n" + "=" * 60)
print("仮説検証")
print("=" * 60)

baseline_score = analysis_baseline['overall'] if analysis_baseline else 0
rag_score = analysis_rag['overall'] if analysis_rag else 0
ft_base_score = analysis_ft_base['overall'] if analysis_ft_base else 0
ft_rag_score = analysis_ft_rag['overall'] if analysis_ft_rag else 0

print("\nH1: FTモデルはL1-L2でRAGと同等以上の性能を達成できる")
if analysis_ft_base and analysis_rag:
    ft_l1l2 = (analysis_ft_base['by_level'].get(1, 0) + analysis_ft_base['by_level'].get(2, 0)) / 2
    rag_l1l2 = (analysis_rag['by_level'].get(1, 0) + analysis_rag['by_level'].get(2, 0)) / 2
    result = "支持" if ft_l1l2 >= rag_l1l2 * 0.95 else "棄却"
    print(f"  FT L1-L2平均: {ft_l1l2:.1f}pt, RAG L1-L2平均: {rag_l1l2:.1f}pt → {result}")
else:
    print("  検証不可（FTモデルなし）")

print("\nH2: FTモデルはL4-L5でRAGに劣る")
if analysis_ft_base and analysis_rag:
    ft_l4l5 = (analysis_ft_base['by_level'].get(4, 0) + analysis_ft_base['by_level'].get(5, 0)) / 2
    rag_l4l5 = (analysis_rag['by_level'].get(4, 0) + analysis_rag['by_level'].get(5, 0)) / 2
    result = "支持" if ft_l4l5 < rag_l4l5 else "棄却"
    print(f"  FT L4-L5平均: {ft_l4l5:.1f}pt, RAG L4-L5平均: {rag_l4l5:.1f}pt → {result}")
else:
    print("  検証不可（FTモデルなし）")

print("\nH3: RAG + FTの組み合わせが最高性能を達成する")
if analysis_ft_rag:
    best_score = max(baseline_score, rag_score, ft_base_score, ft_rag_score)
    result = "支持" if ft_rag_score == best_score else "棄却"
    print(f"  Baseline: {baseline_score:.1f}pt, RAG: {rag_score:.1f}pt, FT-Base: {ft_base_score:.1f}pt, FT+RAG: {ft_rag_score:.1f}pt → {result}")
else:
    print("  検証不可（FTモデルなし）")

print("\nH4: FTモデルは推論速度が大幅に向上する")
if analysis_ft_base and analysis_rag:
    ft_time = analysis_ft_base['avg_time_sec']
    rag_time = analysis_rag['avg_time_sec']
    speedup = (rag_time - ft_time) / rag_time * 100 if rag_time > 0 else 0
    result = "支持" if speedup >= 50 else "棄却"
    print(f"  FT時間: {ft_time:.2f}s, RAG時間: {rag_time:.2f}s, 高速化: {speedup:.1f}% → {result}")
else:
    print("  検証不可（FTモデルなし）")

## Section 7: 結果保存

In [ ]:
# 7.1 JSON保存
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

evaluation_data = {
    "timestamp": timestamp,
    "base_model": BASE_MODEL,
    "ft_model_dir": FT_MODEL_DIR if FT_AVAILABLE else None,
    "test_cases_count": len(TEST_CASES_V2),
    "analysis": {
        "baseline": analysis_baseline,
        "rag": analysis_rag,
        "ft_base": analysis_ft_base,
        "ft_rag": analysis_ft_rag
    },
    "detailed_results": {
        "baseline": baseline_results,
        "rag": rag_results,
        "ft_base": ft_base_results,
        "ft_rag": ft_rag_results
    }
}

result_path = f"{RESULTS_DIR}/finetuning_eval_{timestamp}.json"
with open(result_path, "w", encoding="utf-8") as f:
    json.dump(evaluation_data, f, ensure_ascii=False, indent=2)
print(f"結果保存: {result_path}")

In [ ]:
# 7.2 レポート生成
report = f"""# ファインチューニング性能比較実験 結果レポート

**実行日時**: {timestamp}
**ベースモデル**: {BASE_MODEL}
**テストケース数**: {len(TEST_CASES_V2)}件

## 1. 全体スコア比較

| モデル | スコア | 平均時間(秒) |
|--------|--------|-------------|
| Baseline | {analysis_baseline['overall']:.1f}pt | {analysis_baseline['avg_time_sec']:.2f}s |
| RAG | {analysis_rag['overall']:.1f}pt | {analysis_rag['avg_time_sec']:.2f}s |
"""

if analysis_ft_base:
    report += f"| FT-Base | {analysis_ft_base['overall']:.1f}pt | {analysis_ft_base['avg_time_sec']:.2f}s |\n"
if analysis_ft_rag:
    report += f"| FT+RAG | {analysis_ft_rag['overall']:.1f}pt | {analysis_ft_rag['avg_time_sec']:.2f}s |\n"

report += f"""
## 2. レベル別スコア

| レベル | Baseline | RAG |"""
if analysis_ft_base:
    report += " FT-Base |"
if analysis_ft_rag:
    report += " FT+RAG |"
report += "\n|--------|----------|-----|"
if analysis_ft_base:
    report += "---------|\n"
if analysis_ft_rag:
    report += "--------|\n"

for level in [1, 2, 3, 4, 5]:
    row = f"| L{level} | {analysis_baseline['by_level'].get(level, 'N/A')} | {analysis_rag['by_level'].get(level, 'N/A')} |"
    if analysis_ft_base:
        row += f" {analysis_ft_base['by_level'].get(level, 'N/A')} |"
    if analysis_ft_rag:
        row += f" {analysis_ft_rag['by_level'].get(level, 'N/A')} |"
    report += row + "\n"

report += f"""
## 3. 結論

### 主要な発見

- RAGシステム（Phase 6）は {analysis_rag['overall']:.1f}pt のスコアを達成
- ベースラインは {analysis_baseline['overall']:.1f}pt （RAGなし）
"""

if analysis_ft_base:
    report += f"- ファインチューニングモデル（RAGなし）は {analysis_ft_base['overall']:.1f}pt\n"
if analysis_ft_rag:
    report += f"- ファインチューニング + RAG は {analysis_ft_rag['overall']:.1f}pt\n"

report += f"""
### Phase 7以降への提言

（実験結果に基づいて記述）

---
Generated with Claude Code
"""

report_path = f"{RESULTS_DIR}/finetuning_report_{timestamp}.md"
with open(report_path, "w", encoding="utf-8") as f:
    f.write(report)
print(f"レポート保存: {report_path}")
print("\n" + report)

## Section 8: 完了

In [ ]:
# 8.1 サマリー
print("="*60)
print("ファインチューニング性能比較実験 完了")
print("="*60)

print(f"\n評価モデル数: {len([a for a in [analysis_baseline, analysis_rag, analysis_ft_base, analysis_ft_rag] if a])}")
print(f"テストケース数: {len(TEST_CASES_V2)}件")

print(f"\n結果ファイル:")
print(f"  - {result_path}")
print(f"  - {report_path}")
print(f"  - {RESULTS_DIR}/finetuning_comparison.png")

In [ ]:
# 8.2 メモリ解放
del base_model
if FT_AVAILABLE:
    del ft_model
clear_memory()
print("メモリ解放完了")
print_memory()